# Friendly heatmap builder

This notebook runs `heatmap.py` without editing the plotting code. Upload the helper script and your CSV/XLSX files, choose the data layout below, then run the plotting cells.

In [ ]:
%pip -q install pandas numpy matplotlib openpyxl

## 1. Upload files

Upload `heatmap.py` from this folder and the experiment files you want to plot. You can upload CSV, TSV, XLS or XLSX files.

In [ ]:
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", ", ".join(uploaded.keys()) or "nothing")
except Exception:
    print("This upload button works in Google Colab. Locally, place files next to the notebook.")

print("\nFiles in the current runtime:")
for path in sorted(Path(".").glob("*")):
    if path.is_file():
        print(" -", path.name)

## 2. Choose data settings

For most new experiments, use `one long/tidy table` if your data has one row per measurement, or `one wide dose table` if doses are separate columns like `3_Gy`, `6_Gy`, `9_Gy`.

In [ ]:
#@title Data files and format
mode = "current example: two Excel pH files + optional manual CSV" #@param ["current example: two Excel pH files + optional manual CSV", "one long/tidy table", "one wide dose table", "Excel triplet files listed as file=pH"]

excel_ph_65 = "Protons BiCe GdEuF3 6_5.xlsx" #@param {type:"string"}
ph_65 = 6.5 #@param {type:"number"}
excel_ph_74 = "Protons BiCe GdEuF3 7_4.xlsx" #@param {type:"string"}
ph_74 = 7.4 #@param {type:"number"}
manual_wide_csv = "gdf3_gdeuf3_manual.csv" #@param {type:"string"}

single_table_file = "" #@param {type:"string"}
single_table_family = "All samples" #@param {type:"string"}
excel_files_with_ph = "" #@param {type:"string"}

# Examples: BiCe=BiCe; GdEu=GdF3 / GdEuF3; HeLa=HeLa cells
family_rules = "BiCe=BiCe; GdEu=GdF3 / GdEuF3; GdF3=GdF3 / GdEuF3" #@param {type:"string"}
default_family = "All samples" #@param {type:"string"}

#@title Plot appearance
output_dir = "heatmap_outputs" #@param {type:"string"}
value_label = "$\\ln(\\mathrm{DEF}_{\\mathrm{ROS}})$" #@param {type:"string"}
dose_label = "Dose, Gy" #@param {type:"string"}
concentration_label = "Conc, mg/mL" #@param {type:"string"}
panel_label = "pH" #@param {type:"string"}
colorbar_note = "blue: ROS decrease vs control; red: ROS increase vs control" #@param {type:"string"}
color_map = "bwr" #@param ["bwr", "coolwarm", "RdBu_r", "viridis", "magma", "plasma"]
center_colors_at_zero = True #@param {type:"boolean"}
use_one_color_scale_for_all_groups = True #@param {type:"boolean"}
show_numbers_in_cells = True #@param {type:"boolean"}


In [ ]:
import importlib
import json
from pathlib import Path

if not Path("heatmap.py").exists():
    raise FileNotFoundError("Upload heatmap.py in step 1 before running this cell.")

import heatmap
importlib.reload(heatmap)

def parse_family_rules(text):
    rules = []
    for chunk in text.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "=" not in chunk:
            raise ValueError(f"Family rule must look like contains=family: {chunk}")
        contains, family = chunk.split("=", 1)
        rules.append({"contains": contains.strip(), "family": family.strip()})
    return rules

def parse_excel_file_list(text):
    sources = []
    for chunk in text.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "=" not in chunk:
            raise ValueError(f"Excel entry must look like file.xlsx=6.5: {chunk}")
        path, ph = chunk.split("=", 1)
        sources.append({"path": path.strip(), "format": "triplet_excel", "ph": float(ph.strip())})
    return sources

def file_exists(path):
    return bool(path) and Path(path).exists()

sources = []
if mode == "current example: two Excel pH files + optional manual CSV":
    if file_exists(excel_ph_65):
        sources.append({"path": excel_ph_65, "format": "triplet_excel", "ph": ph_65, "include_families": ["BiCe"]})
    if file_exists(excel_ph_74):
        sources.append({"path": excel_ph_74, "format": "triplet_excel", "ph": ph_74, "include_families": ["BiCe"]})
    if file_exists(manual_wide_csv):
        sources.append({"path": manual_wide_csv, "format": "wide_dose", "family": "GdF3 / GdEuF3"})
elif mode == "one long/tidy table":
    sources.append({"path": single_table_file, "format": "long", "family": single_table_family})
elif mode == "one wide dose table":
    sources.append({"path": single_table_file, "format": "wide_dose", "family": single_table_family})
elif mode == "Excel triplet files listed as file=pH":
    sources.extend(parse_excel_file_list(excel_files_with_ph))

if not sources:
    raise ValueError("No input sources were configured. Check uploaded file names and selected mode.")

config = {
    "sources": sources,
    "family_rules": parse_family_rules(family_rules),
    "default_family": default_family,
    "plot": {
        "output_dir": output_dir,
        "value_label": value_label,
        "colorbar_note": colorbar_note,
        "dose_label": dose_label,
        "concentration_label": concentration_label,
        "panel_label": panel_label,
        "cmap": color_map,
        "center": 0 if center_colors_at_zero else None,
        "global_color_scale": use_one_color_scale_for_all_groups,
        "annotate": show_numbers_in_cells,
    },
}

print(json.dumps(config, indent=2, ensure_ascii=False))

## 3. Build heatmaps

In [ ]:
from IPython.display import Image, display

df = heatmap.load_dataset(config)
print(f"Loaded rows: {len(df)}")
display(df.head(20))

paths = heatmap.plot_heatmaps(df, config["plot"])
for path in paths:
    print(path)
    display(Image(filename=str(path)))

## 4. Download results

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path(output_dir) / "heatmaps.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path, arcname=Path(path).name)

print("Created", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download button is available in Google Colab. Locally, open the ZIP path above.")